In [15]:
import numpy as np
from collections import Counter

class Node:
  def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value
    
class DecisionTreeScratch:
  def __init__(self, min_samples_split=2, max_depth=float("inf"), n_feats=None):
    self.min_samples_split = min_samples_split
    self.max_depth = max_depth
    self.n_feats = n_feats
    self.root = None
    
  def fit(self, X, y):
    self.n_feats = X.shape[1] if not self.n_feats else min(self.n_feats, X.shape[1])
    self.root = self._grow_tree(X, y)
    
  def _grow_tree(self, X , y, depth=0):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))
    
    # 1. Stopping Criteria
    # If pure (1 label), or max depth reached, or too few samples -> Make Leaf
    
    if(depth >=self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
      leaf_value = self._most_comman_label(y)
      return Node(value=leaf_value)
    
    # 2. Find the Best Split
    feat_idx, threshold = self._best_split(X, y, n_features)
    
    # 3. Create Child Nodes
    # If we couldn't find a split (rare), return leaf
    if feat_idx is None:
      return Node(value=self._most_comman_label(y))
    
    left_idxs = X[ :, feat_idx] < threshold
    right_idxs = ~left_idxs # Inverse boolean mask
    
    left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
    right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)
    
    return Node(feature=feat_idx, threshold=threshold, left=left, right=right)
  
  def _best_split(self, X, y, n_features):
    best_gain = -1
    split_idx, split_threshold = None, None
    
    # --- NICHE: Feature Randomness ---
    # Instead of checking 0 to n_features, we pick a random subset.
    # usually sqrt(n_features) is the standard heuristic.
    
    # Assuming self.n_feats is set in __init__ (e.g., int(sqrt(n_features)))
    # If not set, default to all features (Standard Decision Tree)
    
    n_features_total = X.shape[1]
    n_feats_to_check = self.n_feats if hasattr(self, 'n_feats') else n_features_total
    
    feat_idxs = np.random.choice(n_features_total, n_feats_to_check, replace=False)
    
    for feature_idx in range(n_features):
      X_column = X[:, feature_idx]
      thresholds = np.unique(X_column)
      
      for thr in thresholds:
        gain = self._information_gain(y, X_column, thr)
        
        if gain > best_gain:
          best_gain = gain
          split_idx = feature_idx
          split_threshold = thr
          
    return split_idx, split_threshold
  
  def _information_gain(self, y, X_column, threshold):
    # Parent Entropy
    parent_entropy = self._entropy(y)
    
    # Generate Split
    left_idxs = X_column < threshold
    right_idxs = ~left_idxs
    
    if(len(y[left_idxs])==0 or len(y[right_idxs])==0):
      return 0
    
    # Weighted Average Child Entropy
    n = len(y)
    n_l, n_r = len(y[left_idxs]) , len(y[right_idxs])
    e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
    
    child_entropy = (n_l/n) * e_l + (n_r/n) * e_r
    
    return parent_entropy - child_entropy
  
  def _entropy(self, y):
    hist = np.bincount(y)
    ps = hist/len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])
  
  def _most_comman_label(self, y):
    if len(y) ==0: return 0
    counter = Counter(y)
    return counter.most_common(1)[0][0]
  
  def predict(self, X):
    return np.array([self._traverse_tree(x, self.root) for x in X])
  
  def _traverse_tree(self, x, node):
    if node.value is not None:
      return node.value
    
    if x[node.feature] < node.threshold:
      return self._traverse_tree(x, node.left)
    
    return self._traverse_tree(x, node.right)

In [16]:
class RandomForestScratch:
  def __init__(self, n_trees=10, min_samples_split=2, max_depth=100, n_feats=None):
    self.n_trees = n_trees
    self.min_samples_split = min_samples_split
    self.max_depth = max_depth
    self.n_feats = n_feats
    self.trees = []
    
  def fit(self, X, y):
    self.trees = []
    for _ in range(self.n_trees):
      tree = DecisionTreeScratch(
        min_samples_split=self.min_samples_split, 
        max_depth=self.max_depth,
        n_feats=self.n_feats
      )
      
      # Inject the n_feats parameter if you added the Tweak above
      tree.n_feats = self.n_feats
      
      # 2. Bootstrapping (Row Randomness)
      # We create a random dataset of the same size, WITH replacement.
      # This means some rows appear twice, some never (Out-Of-Bag).
      
      X_sample, y_sample = self._bootstrap_sample(X, y)
      
      # 3. Train the tree on this random 'view' of the world
      tree.fit(X_sample, y_sample)
      self.trees.append(tree)
      
  def _bootstrap_sample(self, X, y):
    n_samples = X.shape[0]
    
    idxs = np.random.choice(n_samples, size=n_samples, replace=True)
    return X[idxs], y[idxs]
  
  def _most_comman_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]
  
  def predict(self, X):
    
    tree_preds = np.array([tree.predict(X) for tree in self.trees])
    
    y_pred = [self._most_comman_label(tree_pred) for tree_pred in tree_preds.T]
    
    return np.array(y_pred)

In [53]:
if __name__ == "__main__":
    # np.random.seed(42)
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    # A. Setup Data
    X, y = make_classification(n_samples=100, n_features=10, n_informative=5, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # B. Define Model
    # Niche: n_feats=int(np.sqrt(10)) is the standard heuristic for Classification forests
    forest = RandomForestScratch(n_trees=100, max_depth=20, n_feats=int(np.sqrt(X.shape[1])))

    # C. Train
    print("Training Random Forest...")
    forest.fit(X_train, y_train)

    # D. Predict
    preds = forest.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print(f"Random Forest Accuracy: {acc * 100:.2f}%")

Training Random Forest...
Random Forest Accuracy: 80.00%
